# 08 — Auditoría descriptiva inicial del fenotipo

Este notebook resume multiplicidad de pares, cobertura SOFA y sensibilidad de baseline sobre MIMIC-IV Demo v2.2. Sus cifras son controles de ingeniería, no estimaciones clínicas. Solo muestra agregados; nunca imprime filas ni identificadores.

In [ ]:
from pathlib import Path
import shutil, subprocess, sys, tempfile
import pandas as pd
from IPython.display import SVG, display
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / 'src')) if str(PROJECT_ROOT / 'src') not in sys.path else None
from mimic_sepsis.artifacts import ArtifactStore
from mimic_sepsis.phenotype_audit import coverage_summary, pair_multiplicity, phenotype_summary
manifests = sorted((PROJECT_ROOT / 'data/derived/sofa').glob('*/40_labels/sepsis_episodes.manifest.json'))
valid = []
for path in manifests:
    store = ArtifactStore(path.parent)
    try:
        manifest = store.validate('sepsis_episodes')
        valid.append((manifest.created_at_utc, str(path), store))
    except Exception:
        pass
if not valid:
    raise RuntimeError('Ejecute primero: python scripts/build_demo_sofa_incremental.py --stage all')
_, _, store = sorted(valid)[-1]
pairs = store.read_dataframe('suspected_infection_pairs')
episodes = store.read_dataframe('sepsis_episodes')
stays = store.read_dataframe('sepsis_stays')
shock_stays = store.read_dataframe('septic_shock_stays')


## Tablas agregadas con denominador explícito

In [ ]:
summary = phenotype_summary(pairs, episodes, stays, shock_stays)
multiplicity = pair_multiplicity(pairs)
coverage = coverage_summary(episodes)
display(summary, multiplicity, coverage)


## Figuras con ggplot2

Las figuras se generan con R/ggplot2 a partir de las tablas agregadas anteriores. Se guardan bajo `data/derived/reports/figures`, fuera de Git.

In [ ]:
rscript = shutil.which('Rscript')
if not rscript: raise RuntimeError('Rscript no está disponible en el entorno.')
figure_dir = PROJECT_ROOT / 'data/derived/reports/figures'
figure_dir.mkdir(parents=True, exist_ok=True)
r_code = r'''
args <- commandArgs(trailingOnly=TRUE)
suppressPackageStartupMessages(library(ggplot2))
m <- read.csv(args[1], check.names=FALSE); c <- read.csv(args[2], check.names=FALSE)
p1 <- ggplot(m, aes(pairs_per_admission, admissions)) + geom_col(fill='#2878B5') +
  labs(title='Multiplicidad de pares por ingreso', x='Pares por ingreso', y='Ingresos (n)') + theme_minimal(base_size=12)
p2 <- ggplot(c, aes(coverage, percent)) + geom_col(fill='#D9534F') +
  labs(title='Cobertura temporal del fenotipo', x=NULL, y='Filas par–estancia (%)') +
  theme_minimal(base_size=12) + theme(axis.text.x=element_text(angle=25, hjust=1))
ggsave(args[3], p1, width=7, height=4.5, device=grDevices::svg)
ggsave(args[4], p2, width=8, height=4.8, device=grDevices::svg)
'''
with tempfile.TemporaryDirectory() as tmp:
    tmp = Path(tmp); m_csv = tmp/'multiplicity.csv'; c_csv = tmp/'coverage.csv'
    multiplicity.to_csv(m_csv, index=False); coverage.to_csv(c_csv, index=False)
    p1 = figure_dir/'08_pair_multiplicity.svg'; p2 = figure_dir/'08_sofa_coverage.svg'
    run = subprocess.run([rscript, '-e', r_code, str(m_csv), str(c_csv), str(p1), str(p2)], capture_output=True, text=True)
    if run.returncode: raise RuntimeError(run.stderr)
display(SVG(filename=str(p1)), SVG(filename=str(p2)))


## Interpretación y criterio para avanzar

La unidad `pair` no equivale a episodio independiente: se debe revisar la deduplicación antes de comunicar incidencia. La sensibilidad de baseline y la cobertura se interpretan por estancia y por fila par–estancia, respectivamente. Antes de MIMIC-IV completo deben cerrarse D004 y D011 en `docs/decision_register.md`.